In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import openmc
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# User Settings
Make_Cross_Sections = 'Y'     # 'Y' to draw the cross sections, 'N' to skip them

Chain_File = '/home/tytre/nuclear_data/chain_endfb71_pwr.xml'

# 'specific_activity' keeps the slug size fixed and solves for the Ci/g that gives Target_Dose
# from the bare stack. 'none' uses Specific_Activity as entered.
Solve_For = 'specific_activity'
Target_Dose = 10000.0         # mrem/hr
Specific_Activity = 250.0     # Ci/g, only used when Solve_For = 'none' (INL ATR HSA spec)

# Cobalt slug (1 mm x 1 mm)
Co_Diameter = 0.10            # cm
Co_Height = 0.10              # cm
Co_Density = 8.9              # g/cm3

# Aluminum 6061 capsule
Cap_Diameter, Cap_Height = 3.24, 4.00                 # cm, outside
Cap_Inner_Diameter, Cap_Inner_Height = 2.92, 3.68     # cm, cavity

Pack_Gap = 0.01               # cm between neighboring capsules
Num_Layers = 7
Layer_Gap = 0.0               # cm between layers

# Cask: SS304 / Lead / SS304 (end caps use the same thicknesses)
Cask_Inner_Radius = 8.0       # cm
SS_Inner = 0.3715             # cm
Lead_Thickness = 10.49         # cm
SS_Outer = 0.3175             # cm
Axial_Clearance = 0.1         # cm between the stack ends and the cavity ends
Target_Dist = 30.0            # cm from the cask surface
Sizing_Dist = 30.0            # cm from the bare stack surface (source sizing only)
Max_Dist = 50.0               # cm past the cask surface covered by the mesh

# Cross section plots
Plot_Width = Plot_Height = 42.0       # cm, radial view
Plot_Res = 180                        # pixels per cm
Side_Width, Side_Height = 42.0, 56.0  # cm, axial view
Side_Res = 25

Batches = 50
Particles = 500_000           # per batch

openmc.config['chain_file'] = Chain_File

Cap_R, Cav_R = Cap_Diameter / 2, Cap_Inner_Diameter / 2
Co_R = Co_Diameter / 2
assert Co_R < Cav_R and Co_Height < Cap_Inner_Height, "Co slug does not fit in the capsule"

# Dose Coefficients (ICRP 116, AP geometry; energies in eV, pSv-cm2)
print('Loading dose coefficients and Co-60 decay data...', flush=True)
energies, coeffs = openmc.data.dose_coefficients('photon', geometry='AP', data_source='icrp116')

# Pure Co-60 Reference
pure = openmc.Material()
pure.add_nuclide('Co60', 1.0, 'wo')
pure.set_density('g/cm3', Co_Density)
pure.volume = 1.0
pure_bq_per_g = pure.get_activity(units='Bq/g')
pure_ci_per_g = pure_bq_per_g / 3.7e10      # about 1130 Ci/g

spec = pure.get_decay_photon_energy(units='Bq/cm3')
E, I = np.asarray(spec.x, float), np.asarray(spec.p, float)
dose_per_decay = (I * np.interp(E, energies, coeffs)).sum() / (pure_bq_per_g * Co_Density)  # pSv-cm2

# 1 Ci point source at 1 m, in rem-m2/(hr-Ci)   (1 pSv = 1e-7 mrem)
gamma_eff = 3.7e10 * dose_per_decay / (4 * math.pi * 1.0e4) * 3600 * 1e-7 / 1000.0

# Capsule Layout (optimal packing of 19 circles)
# R = (1 + sqrt2 + sqrt6) r: one in the center, 6 at 2r (rotated 15 deg), 12 at (sqrt2 + sqrt6) r
r_eff = Cap_R + Pack_Gap / 2
inner_r, outer_r = 2.0 * r_eff, (math.sqrt(2.0) + math.sqrt(6.0)) * r_eff
Array_R = outer_r + Cap_R       # radius enclosing all capsules

centers = [(0.0, 0.0)]

for k in range(6):
    angle = math.radians(15.0 + 60.0 * k)
    centers.append((inner_r * math.cos(angle), inner_r * math.sin(angle)))

for k in range(12):
    angle = math.radians(30.0 * k)
    centers.append((outer_r * math.cos(angle), outer_r * math.sin(angle)))

Num_Caps = len(centers)

layer_z = [(k - (Num_Layers - 1) / 2.0) * (Cap_Height + Layer_Gap) for k in range(Num_Layers)]
Stack_Height = Num_Layers * Cap_Height + (Num_Layers - 1) * Layer_Gap
Num_Slugs = Num_Caps * Num_Layers

# Source Sizing (bare stack dose at Sizing_Dist = Target_Dose)
print('Sizing the source and building materials and geometry...', flush=True)
obs_r = Array_R + Sizing_Dist
inv_d2 = sum(1.0e4 / ((obs_r - x)**2 + y**2 + z**2) for x, y in centers for z in layer_z)  # 1/m2
mrem_hr_per_ci = gamma_eff * 1000.0 * inv_d2    # dose per Ci in each slug

slug_volume = math.pi * Co_R**2 * Co_Height
slug_mass = slug_volume * Co_Density
if Solve_For == 'specific_activity':
    Specific_Activity = Target_Dose / mrem_hr_per_ci / slug_mass
co60_wo = Specific_Activity / pure_ci_per_g     # Co-60 mass fraction
assert 0.0 < co60_wo <= 1.0, f"Specific activity must be between 0 and {pure_ci_per_g:.0f} Ci/g"

# Materials
# Natural Co for transport (photon interactions only depend on the element)
co_transport = openmc.Material(name='Co Slug (Transport)')
co_transport.add_element('Co', 1.0)
co_transport.set_density('g/cm3', Co_Density)

# Co-60 in Co-59, only used for the decay photon spectrum
co60_source = openmc.Material(name='Co-60 / Co-59 (Source)')
co60_source.add_nuclide('Co60', co60_wo, 'wo')
co60_source.add_nuclide('Co59', 1.0 - co60_wo, 'wo')
co60_source.set_density('g/cm3', Co_Density)
co60_source.volume = slug_volume
activity_Bq = co60_source.get_activity(units='Bq')    # per slug
activity_Ci = activity_Bq / 3.7e10

# Dry air, 20 C
air = openmc.Material(name='Air')
air.add_element('N', 0.755093, 'wo')
air.add_element('O', 0.231847, 'wo')
air.add_element('Ar', 0.01288, 'wo')
air.add_element('C', 0.000175, 'wo')
air.add_element('Kr', 0.000003, 'wo')
air.add_element('He', 0.000001, 'wo')
air.add_element('H', 0.0000003, 'wo')
air.add_element('Xe', 0.0000004, 'wo')
air.set_density('g/cm3', 1.205e-3)

# Aluminum 6061-T6
aluminum = openmc.Material(name='Aluminum')
aluminum.add_element('Al', 0.97305, 'wo')
aluminum.add_element('Mg', 0.01, 'wo')
aluminum.add_element('Si', 0.006, 'wo')
aluminum.add_element('Fe', 0.0035, 'wo')
aluminum.add_element('Cu', 0.00275, 'wo')
aluminum.add_element('Cr', 0.00195, 'wo')
aluminum.add_element('Zn', 0.00125, 'wo')
aluminum.add_element('Mn', 0.00075, 'wo')
aluminum.add_element('Ti', 0.00075, 'wo')
aluminum.set_density('g/cm3', 2.7)

# SS304
ss304 = openmc.Material(name='SS304')
ss304.add_element('Fe', 0.695, 'wo')
ss304.add_element('Cr', 0.19, 'wo')
ss304.add_element('Ni', 0.1, 'wo')
ss304.add_element('Mn', 0.01, 'wo')
ss304.add_element('Si', 0.005, 'wo')
ss304.set_density('g/cm3', 8.0)

# Lead
lead = openmc.Material(name='Lead')
lead.add_element('Pb', 1.0)
lead.set_density('g/cm3', 11.34)

materials = openmc.Materials([co_transport, air, aluminum, ss304, lead])

# Cask Dimensions
assert Cask_Inner_Radius > Array_R, f"Cask cavity must be larger than the stack ({Array_R:.3f} cm)"
R0 = Cask_Inner_Radius
R1 = R0 + SS_Inner          # inner SS304 / lead
R2 = R1 + Lead_Thickness    # lead / outer SS304
R3 = R2 + SS_Outer          # cask surface
H0 = Stack_Height / 2 + Axial_Clearance     # cavity half height
H1 = H0 + SS_Inner
H2 = H1 + Lead_Thickness
H3 = H2 + SS_Outer
r_target = R3 + Target_Dist

# Geometry
outer = openmc.Sphere(r=R3 + Max_Dist + 20.0, boundary_type='vacuum')

# Shared z-planes (with Layer_Gap = 0 the top of one layer is the bottom of the next)
planes = {}
def zplane(z):
    key = round(z, 9)
    if key not in planes:
        planes[key] = openmc.ZPlane(z0=key)
    return planes[key]

# One set of cylinders per (x, y) position, reused for every layer
co_cyls, cav_cyls, cap_cyls = [], [], []
for cx, cy in centers:
    co_cyls.append(openmc.ZCylinder(x0=cx, y0=cy, r=Co_R))
    cav_cyls.append(openmc.ZCylinder(x0=cx, y0=cy, r=Cav_R))
    cap_cyls.append(openmc.ZCylinder(x0=cx, y0=cy, r=Cap_R))

slug_cells, cavity_cells, wall_cells, cap_regions = [], [], [], []

for L, zc in enumerate(layer_z, start=1):
    co_bottom = zplane(zc - Co_Height / 2)
    co_top = zplane(zc + Co_Height / 2)
    cav_bottom = zplane(zc - Cap_Inner_Height / 2)
    cav_top = zplane(zc + Cap_Inner_Height / 2)
    cap_bottom = zplane(zc - Cap_Height / 2)
    cap_top = zplane(zc + Cap_Height / 2)

    for i in range(Num_Caps):
        co_region = -co_cyls[i] & +co_bottom & -co_top
        cav_region = -cav_cyls[i] & +cav_bottom & -cav_top
        cap_region = -cap_cyls[i] & +cap_bottom & -cap_top

        slug = openmc.Cell(name=f'Slug L{L}-{i + 1}')
        slug.fill = co_transport
        slug.region = co_region
        slug_cells.append(slug)

        cavity = openmc.Cell(name=f'Cavity L{L}-{i + 1}')
        cavity.fill = air
        cavity.region = cav_region & ~co_region
        cavity_cells.append(cavity)

        wall = openmc.Cell(name=f'Wall L{L}-{i + 1}')
        wall.fill = aluminum
        wall.region = cap_region & ~cav_region
        wall_cells.append(wall)

        cap_regions.append(cap_region)

# Cask layers as nested closed cylinders centered on z = 0
cask_cavity_region = -openmc.ZCylinder(r=R0) & +zplane(-H0) & -zplane(H0)
ss_inner_region = -openmc.ZCylinder(r=R1) & +zplane(-H1) & -zplane(H1)
lead_region = -openmc.ZCylinder(r=R2) & +zplane(-H2) & -zplane(H2)
ss_outer_region = -openmc.ZCylinder(r=R3) & +zplane(-H3) & -zplane(H3)

# Air in the cask cavity outside the capsules
cask_air_region = cask_cavity_region
for cap_region in cap_regions:
    cask_air_region = cask_air_region & ~cap_region
cask_air_cell = openmc.Cell(name='Cask Cavity Air')
cask_air_cell.fill = air
cask_air_cell.region = cask_air_region

ss1_cell = openmc.Cell(name='Cask SS304 Inner')
ss1_cell.fill = ss304
ss1_cell.region = ss_inner_region & ~cask_cavity_region

pb_cell = openmc.Cell(name='Cask Lead')
pb_cell.fill = lead
pb_cell.region = lead_region & ~ss_inner_region

ss2_cell = openmc.Cell(name='Cask SS304 Outer')
ss2_cell.fill = ss304
ss2_cell.region = ss_outer_region & ~lead_region

air_cell = openmc.Cell(name='Outside Air')
air_cell.fill = air
air_cell.region = -outer & ~ss_outer_region

geometry = openmc.Geometry(slug_cells + cavity_cells + wall_cells
                           + [cask_air_cell, ss1_cell, pb_cell, ss2_cell, air_cell])

# Cross Section Plots (radial x-y and axial x-z)
def surface_value(s, X, Y, Z):
    if isinstance(s, openmc.ZCylinder):
        return (X - s.x0)**2 + (Y - s.y0)**2 - s.r**2
    if isinstance(s, openmc.ZPlane):
        return Z - s.z0
    if isinstance(s, openmc.Sphere):
        return (X - s.x0)**2 + (Y - s.y0)**2 + (Z - s.z0)**2 - s.r**2
    return s.evaluate((X, Y, Z))


def region_mask(reg, X, Y, Z):
    """True wherever (X, Y, Z) is inside the region."""
    if isinstance(reg, openmc.Halfspace):
        v = surface_value(reg.surface, X, Y, Z)
        return v >= 0 if reg.side == '+' else v < 0
    if isinstance(reg, openmc.Intersection):
        m = np.ones(X.shape, dtype=bool)
        # check z-planes first since they are cheap, stop once nothing is left
        for node in sorted(reg, key=lambda n: 0 if isinstance(getattr(n, 'surface', None),
                                                              openmc.ZPlane) else 1):
            m &= region_mask(node, X, Y, Z)
            if not m.any():
                break
        return m
    if isinstance(reg, openmc.Union):
        m = np.zeros(X.shape, dtype=bool)
        for node in reg:
            m |= region_mask(node, X, Y, Z)
        return m
    if isinstance(reg, openmc.Complement):
        return ~region_mask(reg.node, X, Y, Z)
    raise TypeError(f"Unsupported region type: {type(reg)}")


def plot_section(cells, layers, labels, colors, plane, width, height, px_per_cm, fname):
    """Draw a 2D slice of the geometry. plane = 'xy' (z = 0) or 'xz' (y = 0).
    layers[i] is the color index for cells[i], -1 means don't draw it."""
    nh, nv = int(round(width * px_per_cm)), int(round(height * px_per_cm))
    h0, h1, v0, v1 = -width / 2, width / 2, -height / 2, height / 2
    H, V = np.meshgrid(h0 + (np.arange(nh) + 0.5) * width / nh,
                       v0 + (np.arange(nv) + 0.5) * height / nv)
    F = np.zeros_like(H)
    X, Y, Z = (H, V, F) if plane == 'xy' else (H, F, V)

    ids = np.full((nv, nh), -1, dtype=int)      # color index per pixel
    cid = np.full((nv, nh), -1, dtype=int)      # cell index per pixel
    print(f'Drawing the {plane} cross section...', flush=True)
    for n, (cell, k) in enumerate(zip(cells, layers)):
        m = region_mask(cell.region, X, Y, Z) & (cid < 0)
        ids[m] = k
        cid[m] = n

    fw = 6.5
    fh = fw * height / width
    if fh > 8.5:
        fw, fh = fw * 8.5 / fh, 8.5
    fig, ax = plt.subplots(figsize=(fw, fh + 0.7))
    ext = (h0, h1, v0, v1)
    ax.imshow(np.ma.masked_less(ids, 0), origin='lower', extent=ext,
              cmap=ListedColormap(colors), vmin=-0.5, vmax=len(colors) - 0.5,
              interpolation='nearest')

    # Outline every cell boundary
    edge = np.zeros(cid.shape, dtype=bool)
    edge[:, 1:] |= cid[:, 1:] != cid[:, :-1]
    edge[1:, :] |= cid[1:, :] != cid[:-1, :]
    if px_per_cm >= 60:
        edge = edge | np.roll(edge, 1, axis=0) | np.roll(edge, 1, axis=1)
    ax.imshow(np.ma.masked_where(~edge, edge), origin='lower', extent=ext,
              cmap=ListedColormap(['k']), interpolation='nearest')

    handles = [Patch(fc=colors[k], ec='k', label=labels[k])
               for k in range(len(colors)) if np.any(ids == k)]

    # Scale bar, up to 20% of the view width
    bar = max((b for b in (0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50) if b <= 0.2 * width),
              default=0.2 * width)
    sx, sy = h0 + 0.05 * width, v0 + 0.04 * height
    ax.plot([sx, sx + bar], [sy, sy], 'k-', lw=4, solid_capstyle='butt')
    ax.text(sx + bar / 2, sy + 0.015 * height, f'{bar:g} cm', ha='center', va='bottom',
            fontsize=10, fontweight='bold', bbox=dict(fc='white', ec='none', alpha=0.8, pad=1))

    ax.set_xlim(h0, h1)
    ax.set_ylim(v0, v1)
    ax.set_aspect('equal')
    ax.axis('off')
    fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=9, frameon=False)
    fig.tight_layout(rect=(0, 0.10 * 6.5 / (fh + 0.7), 1, 1))
    fig.savefig(fname, dpi=max(150, px_per_cm * width / fw))


plot_cells = slug_cells + cavity_cells + wall_cells + [cask_air_cell, ss1_cell, pb_cell,
                                                       ss2_cell, air_cell]
plot_layers = ([0] * len(slug_cells) + [1] * len(cavity_cells) + [2] * len(wall_cells)
               + [5, 3, 4, 3, -1])      # outside air is not drawn
plot_labels = ['Cobalt (Co-60 in Co-59)', 'Air (Inside Capsule)', 'Aluminum 6061 Capsule Wall',
               'SS304', 'Lead', 'Air (Inside Cask)']
plot_colors = ['#d9a441', '#f4ecd8', '#9db4c9', '#7d8793', '#3e4a5a', '#eef3f7']

if Make_Cross_Sections.strip().upper() == 'Y':
    plot_section(plot_cells, plot_layers, plot_labels, plot_colors, 'xy', Plot_Width,
                 Plot_Height, Plot_Res, 'co60_geometry.png')
    plot_section(plot_cells, plot_layers, plot_labels, plot_colors, 'xz', Side_Width,
                 Side_Height, Side_Res, 'co60_geometry_side.png')
else:
    print('Skipping the cross section plots')

# Source
print('Building the photon source...', flush=True)
energy_dist = co60_source.get_decay_photon_energy(units='Bq')
total_rate = Num_Slugs * energy_dist.integral()     # photons/s, whole stack

# One uniform source per slug. Strengths sum to 1 since OpenMC normalizes tallies to the
# total strength; the actual photon rate gets applied in post-processing.
sources = []
for zc in layer_z:
    for cx, cy in centers:
        space = openmc.stats.CylindricalIndependent(
            r=openmc.stats.PowerLaw(0.0, Co_R, 1),      # pdf ~ r gives uniform over the disk
            phi=openmc.stats.Uniform(0.0, 2 * math.pi),
            z=openmc.stats.Uniform(-Co_Height / 2, Co_Height / 2),
            origin=(cx, cy, zc),
        )
        source = openmc.IndependentSource()
        source.space = space
        source.angle = openmc.stats.Isotropic()
        source.energy = energy_dist
        source.particle = 'photon'
        source.strength = 1.0 / Num_Slugs
        sources.append(source)

settings = openmc.Settings()
settings.run_mode = 'fixed source'
settings.photon_transport = True
settings.source = sources
settings.batches = Batches
settings.particles = Particles
settings.verbosity = 7

# Tallies (cylindrical r-z mesh, photon flux and ICRP 116 dose)
def layer_edges(a, b, width):
    """Evenly spaced bin edges from a to b, roughly `width` apart."""
    return np.linspace(a, b, max(1, int(round((b - a) / width))) + 1)

# Cavity is one bin, finer bins through the cask walls, then 1 cm bins outside centered on
# whole-cm distances from the cask surface (so Target_Dist lands on a bin center)
r_edges = np.concatenate([[0.0], layer_edges(R0, R1, 0.5), layer_edges(R1, R2, 1.0)[1:],
                          layer_edges(R2, R3, 0.5)[1:],
                          R3 + np.arange(0.5, Max_Dist + 1.0, 1.0)])
z_edges = np.arange(-50.5, 51.0, 1.0)       # 1 cm bins, one centered on z = 0

mesh = openmc.CylindricalMesh(r_grid=r_edges, phi_grid=(0.0, 2 * math.pi), z_grid=z_edges,
                              origin=(0.0, 0.0, 0.0))
mesh_filter = openmc.MeshFilter(mesh)
photon_filter = openmc.ParticleFilter('photon')

flux_tally = openmc.Tally(name='flux mesh')
flux_tally.filters = [mesh_filter, photon_filter]
flux_tally.scores = ['flux']

dose_tally = openmc.Tally(name='dose mesh')
dose_tally.filters = [mesh_filter, photon_filter, openmc.EnergyFunctionFilter(energies, coeffs)]
dose_tally.scores = ['flux']

# Run
model = openmc.Model(geometry, materials, settings, openmc.Tallies([flux_tally, dose_tally]))
print(f'Running OpenMC: {Batches} batches x {Particles:,} particles...', flush=True)
sp_path = model.run(output=True)

# Post-Processing
nr, nz = len(r_edges) - 1, len(z_edges) - 1
with openmc.StatePoint(sp_path) as sp:
    t, td = sp.get_tally(name='flux mesh'), sp.get_tally(name='dose mesh')
    # r varies fastest, so reshape to (nz, nr)
    mean, std = t.mean.reshape(nz, nr), t.std_dev.reshape(nz, nr)
    dmean, dstd = td.mean.reshape(nz, nr), td.std_dev.reshape(nz, nr)

bin_vol = (math.pi * (r_edges[1:]**2 - r_edges[:-1]**2))[None, :] * np.diff(z_edges)[:, None]  # cm3
flux, flux_err = mean / bin_vol * total_rate, std / bin_vol * total_rate      # photons/cm2-s

# pSv-cm2 per source photon to mrem/hr (1 pSv = 1e-6 uSv, 1 uSv = 0.1 mrem)
to_mrem_hr = total_rate * 3600 / 1e6 * 0.1
dose, dose_err = dmean / bin_vol * to_mrem_hr, dstd / bin_vol * to_mrem_hr

r_mid = 0.5 * (r_edges[1:] + r_edges[:-1])
iz0 = int(np.argmin(np.abs(0.5 * (z_edges[1:] + z_edges[:-1]))))    # z = 0 row
rp = r_mid[1:]                                                      # inner wall outward

# Unshielded: sum of 1/d^2 over every slug, at radius r and z = 0
def inv_d2_at(r):
    return sum(1.0 / ((r - x)**2 + y**2 + z**2) for x, y in centers for z in layer_z)   # 1/cm2

flux_no_shield = np.array([energy_dist.integral() * inv_d2_at(r) / (4 * math.pi) for r in rp])
dose_no_shield = np.array([gamma_eff * 1000.0 * activity_Ci * 1.0e4 * inv_d2_at(r) for r in rp])

def plot_vs_distance(y, yerr, no_shield, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.errorbar(rp, y[iz0, 1:], yerr=yerr[iz0, 1:], fmt='o-', ms=3, label='With Cask')
    ax.plot(rp, no_shield, 'k--', label='No Shielding')
    for n, rr in enumerate((R0, R1, R2, R3)):
        ax.axvline(rr, color='tab:red', lw=0.6, alpha=0.6,
                   label='Cask Layer Boundaries' if n == 0 else None)
    ax.axvline(r_target, color='gray', ls=':', label=f'{Target_Dist:g} cm from Cask Surface')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Distance from Z-Axis (cm)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=150)

plot_vs_distance(flux, flux_err, flux_no_shield, 'Photon Flux (photons/cm²-s)',
                 'Co-60 Photon Flux vs. Distance', 'co60_flux_vs_distance.png')
plot_vs_distance(dose, dose_err, dose_no_shield, 'Dose Rate (mrem/hr)',
                 'Co-60 Dose Rate vs. Distance', 'co60_dose_vs_distance.png')

# Results at z = 0
i_in = 1                                            # first bin starts at the inner wall
i_out = int(np.argmin(np.abs(r_mid - r_target)))
print(f"\nSource: {Specific_Activity:.4g} Ci/g, {activity_Ci:.4g} Ci per slug x {Num_Slugs} "
      f"= {activity_Ci * Num_Slugs:.4g} Ci total")
for name, i in (("Inner SS304 wall", i_in),
                (f"{Target_Dist:g} cm from the outer SS304 wall", i_out)):
    print(f"{name} (r = {r_edges[i]:.2f}-{r_edges[i + 1]:.2f} cm): "
          f"dose {dose[iz0, i]:.4e} +/- {dose_err[iz0, i]:.1e} mrem/hr, "
          f"flux {flux[iz0, i]:.4e} photons/cm2-s")
print(f"{Target_Dist:g} cm from the bare stack, no shielding "
      f"(r = {Array_R + Target_Dist:.2f} cm): "
      f"dose {gamma_eff * 1000.0 * activity_Ci * 1.0e4 * inv_d2_at(Array_R + Target_Dist):.4e} mrem/hr")

plt.show()

KeyboardInterrupt: 